# 1. Load and split the data

In [1]:
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

data = load_wine()

X = data.data
y = data.target

# Train: 70%, validation: 15%, test: 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Fit scaler ONLY on training data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(124, 13)
(27, 13)
(27, 13)


# 2. Convert data to PyTorch tensors

In [2]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

# 3. Build the neural network

In [3]:
class WineClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(13, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)


model = WineClassifier()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)

# 4. Train and validate

In [4]:
best_val_loss = float("inf")
best_state = None

for epoch in range(100):
    # Training
    model.train()

    for xb, yb in train_loader:
        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

    # Validation
    model.eval()

    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            k: v.detach().clone()
            for k, v in model.state_dict().items()
        }

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}: "
            f"Validation loss = {val_loss:.4f}"
        )

model.load_state_dict(best_state)

Epoch 10: Validation loss = 0.5592
Epoch 20: Validation loss = 0.2640
Epoch 30: Validation loss = 0.0827
Epoch 40: Validation loss = 0.0367
Epoch 50: Validation loss = 0.0234
Epoch 60: Validation loss = 0.0175
Epoch 70: Validation loss = 0.0139
Epoch 80: Validation loss = 0.0117
Epoch 90: Validation loss = 0.0101
Epoch 100: Validation loss = 0.0089


<All keys matched successfully>

# 5. Test the final model

In [5]:
model.eval()

with torch.no_grad():
    logits = model(X_test)
    predictions = torch.argmax(logits, dim=1)

accuracy = accuracy_score(
    y_test.numpy(),
    predictions.numpy()
)

print("Test accuracy:", round(accuracy, 4))

print(classification_report(
    y_test.numpy(),
    predictions.numpy(),
    target_names=data.target_names
))

Test accuracy: 0.963
              precision    recall  f1-score   support

     class_0       0.90      1.00      0.95         9
     class_1       1.00      0.91      0.95        11
     class_2       1.00      1.00      1.00         7

    accuracy                           0.96        27
   macro avg       0.97      0.97      0.97        27
weighted avg       0.97      0.96      0.96        27



# 6. Save the model and scaler

In [6]:
import joblib

torch.save(model.state_dict(), "wine_model.pth")
joblib.dump(scaler, "wine_scaler.pkl")

['wine_scaler.pkl']

# Load Model + Make Predictions

## 1. Load the saved model


In [7]:
import torch
import torch.nn as nn
import joblib

class WineClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(13, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)


model = WineClassifier()

model.load_state_dict(
    torch.load("wine_model.pth", map_location="cpu", weights_only=True)
)

model.eval()

scaler = joblib.load("wine_scaler.pkl")

print("Model loaded successfully!")

Model loaded successfully!


## 2. Prepare new input data

In [11]:
import numpy as np

# Example: first wine sample from the dataset
new_data = np.array([[
    13.2, 1.78, 2.14, 11.2, 100,
    2.65, 2.76, 0.26, 1.28, 4.38,
    1.05, 3.4, 1050
]])

# Apply the previously fitted scaler
new_data = scaler.transform(new_data)

# Convert to PyTorch tensor
new_tensor = torch.tensor(
    new_data,
    dtype=torch.float32
)

print(new_tensor)

tensor([[ 0.3021, -0.4803, -0.8337, -2.5105, -0.0027,  0.5715,  0.7915, -0.8525,
         -0.5647, -0.2590,  0.4209,  1.1701,  0.9832]])


## 3. Make a prediction

In [12]:
with torch.no_grad():
    logits = model(new_tensor)

    probabilities = torch.softmax(logits, dim=1)

    predicted_class = torch.argmax(
        probabilities, dim=1
    ).item()

print("Predicted class:", predicted_class)
print("Probabilities:", probabilities.numpy())

Predicted class: 0
Probabilities: [[9.9975771e-01 2.1603085e-04 2.6268946e-05]]


## 4. Get the class name

In [13]:
from sklearn.datasets import load_wine

data = load_wine()

class_name = data.target_names[predicted_class]

print("Prediction:", class_name)

Prediction: class_0


## 5. Create a reusable prediction function

In [14]:
def predict_wine(features):
    features = np.array([features])

    scaled = scaler.transform(features)

    tensor = torch.tensor(
        scaled,
        dtype=torch.float32
    )

    with torch.no_grad():
        output = model(tensor)
        probabilities = torch.softmax(output, dim=1)

        predicted = torch.argmax(
            probabilities, dim=1
        ).item()

    return {
        "prediction": data.target_names[predicted],
        "confidence": round(
            probabilities[0, predicted].item(), 4
        )
    }


result = predict_wine([
    13.2, 1.78, 2.14, 11.2, 100,
    2.65, 2.76, 0.26, 1.28, 4.38,
    1.05, 3.4, 1050
])

print(result)

{'prediction': np.str_('class_0'), 'confidence': 0.9998}
